In [ ]:
# import required libraries
import os
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
from urllib.request import urlretrieve
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score

# download face detection model if not present
model_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
model_path = "haarcascade_frontalface_default.xml"

if not os.path.exists(model_path):
    print("Downloading face model")
    urlretrieve(model_url, model_path)

# load face detector
face_cascade = cv2.CascadeClassifier(model_path)

# function to display image
def show_image(img, title="Image"):
    plt.figure(figsize=(8,6))
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()

# detect faces from image
def detect_faces(image_path, save_path=None):

    img = cv2.imread(image_path)

    if img is None:
        print("Image not found")
        return None, 0

    # convert image to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # detect faces
    faces = face_cascade.detectMultiScale(gray,1.1,5)

    features = []

    # draw rectangle around faces
    for (x,y,w,h) in faces:

        cv2.rectangle(img,(x,y),(x+w,y+h),(255,0,0),2)

        # extract face region
        face = gray[y:y+h, x:x+w]

        # resize face
        face = cv2.resize(face,(50,50))

        # flatten features
        features.append(face.flatten())

    # apply minmax scaling
    if len(features) > 0:

        X = np.array(features)

        scaler = MinMaxScaler()

        X_scaled = scaler.fit_transform(X)

        print("Feature shape:", X_scaled.shape)

    else:
        print("No face detected")

    # save result image
    if save_path:
        cv2.imwrite(save_path,img)

    # convert to rgb for plotting
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    return rgb, len(faces)

# evaluate detection using simple metrics
def evaluate(detected, actual=1):

    y_true = [1 if actual>0 else 0]
    y_pred = [1 if detected>0 else 0]

    acc = accuracy_score(y_true,y_pred)
    prec = precision_score(y_true,y_pred, zero_division=0)
    rec = recall_score(y_true,y_pred, zero_division=0)

    print("\nEvaluation Results")
    print("Accuracy:",acc)
    print("Precision:",prec)
    print("Recall:",rec)

# webcam face detection
def webcam_detection():

    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Cannot open webcam")
        return

    print("Press q to exit")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(gray,1.1,5)

        for (x,y,w,h) in faces:
            cv2.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),2)

        cv2.imshow("Face Detection",frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# detect faces in video
def video_detection(video_path, output_path=None):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Cannot open video")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = None

    # create video writer if output required
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(output_path,fourcc,fps,(width,height))

    frame_count = 0
    start = time.time()

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(gray,1.1,5)

        for (x,y,w,h) in faces:
            cv2.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),2)

        if writer:
            writer.write(frame)

        frame_count += 1

        if frame_count % 30 == 0:
            print("Processed",frame_count,"frames")

    cap.release()

    if writer:
        writer.release()

    print("Video processing finished")

# main execution
input_image = "test_image.jpg"
output_image = "result.jpg"

# create sample image if not present
if not os.path.exists(input_image):
    img = np.zeros((500,500,3),dtype=np.uint8)
    cv2.imwrite(input_image,img)

# run face detection
result, faces = detect_faces(input_image, output_image)

if result is not None:
    show_image(result,"Detected Faces")
    evaluate(faces)

# optional
# webcam_detection()
# video_detection("input_video.mp4","output_video.mp4")